# Lab 2: Forecasting Benefit Payouts with XGBoost

**Time:** about 60 minutes

In Lab 1 you forecast the money coming **in**. Now forecast the money going **out**: how much the Fund will pay in
benefits each month. Finance needs this to make sure enough cash is available on the day pensions are paid.

### A different route to a forecast

In Lab 1 the models (SARIMA, Prophet) were built for time series: they looked at the series itself and nothing else.
This lab takes another route. You'll turn forecasting into an ordinary **supervised learning** problem, the kind you
solved on Day 2: a table of features, a target column, and a model that learns how the features map to the target.

The difficulty is that a model like XGBoost has no idea what "time" is. It sees rows, not a sequence. So you give it
the recent past **as columns**. For each month, the row holds what the payouts were 1, 2, 3, 6 and 12 months earlier,
and the target is that month's payout:

```
month    payout   lag_1   lag_3   lag_12   rolling_mean_3   month_of_year ...  →  target: this month's payout
```

Then XGBoost (eXtreme Gradient Boosting) does what it did on Day 2: learns which combinations of those columns predict
the target. On Day 2 it predicted a yes/no (fraud or not); here it predicts a **number** (TZS billions), which is
called **regression**.

### Words you'll meet in this lab

| Word | Plain meaning |
|---|---|
| **supervised learning** | learning from examples where the right answer (the target) is known |
| **regression** | predicting a number (an amount), rather than a category |
| **lag** | the value from an earlier month, e.g. `lag_12` = the payout 12 months ago |
| **rolling mean** | the average of the last few months, e.g. of the last 3 |
| **rolling standard deviation** | how much the last few months jumped around |
| **calendar feature** | something about the date itself: month, quarter, "is it June?" |
| **leakage** | a feature that secretly contains the answer, making the model look better than it is |
| **one step ahead** | forecasting next month only, using the real values up to this month |
| **recursive forecast** | forecasting 12 months by predicting one month, treating it as real, then predicting the next |
| **direct forecast** | a separate model for each distance ahead (1 month, 2 months ...) |
| **global model** | one model trained on several related series at once |
| **feature importance** | a score of how much the model relied on each column |

## Where this lab fits

**What you've done so far.** In Lab 1 you forecast contribution cash coming **in**, with time-series models (SARIMA and
Prophet) and three baselines. The seasonal naive with growth won on 2025. On Day 2 you used XGBoost to score claims.

**What you'll do now.** You'll forecast benefit payouts going **out**, by recasting forecasting as the kind of table
problem XGBoost already knows: each month becomes a row, its recent history becomes columns, and its payout is the
target. You'll test it one month ahead and twelve months ahead, and try a single model across all benefit types.

**Why it matters.** Pensions have to be paid on the day they're due, every month, whatever happens to contributions.
Finance needs to know how much cash to keep ready. Together with Lab 1, this gives both sides of the Fund's monthly cash
flow.

**What happens next.** Lab 3 tests both labs' methods over several years, not one, and turns the results into a
planning pack with honest ranges for finance.

### Theory: forecasting as supervised learning

Any time series can be rewritten as an ordinary table. Take the monthly payouts:

```
Jan 1.70   Feb 1.75   Mar 1.82   Apr 1.79   May 1.91
```

and give each month a row, with the months before it as columns:

| month | lag_1 (last month) | lag_2 (two months ago) | target (this month) |
|---|---|---|---|
| Mar | 1.75 | 1.70 | 1.82 |
| Apr | 1.82 | 1.75 | 1.79 |
| May | 1.79 | 1.82 | 1.91 |

Now it's a standard regression problem: learn how the columns predict the target. Any regression model can do it:
linear regression, XGBoost, even a neural network.

Compared with Lab 1's time-series models:

| | SARIMA / Prophet | XGBoost on lag features |
|---|---|---|
| Knows about time | yes, built in | only through the columns you give it |
| Extra information (calendar, beneficiaries, benefit type) | awkward | easy: add a column |
| Many series at once | one model per series | one model for all (Section 6) |
| Intervals | built in | not built in (Lab 3 adds them) |
| Can extrapolate a trend beyond what it's seen | yes | **no**: trees can only predict values like those in training |

That last row matters for a growing fund. A tree model can't forecast a payout higher than any it has seen, because each
leaf predicts an average of training values. Lag features help, because they carry the current level into each row.

## Imports and settings

Only one new library: `XGBRegressor`, the version of XGBoost that predicts numbers. The rest are the same as Lab 1.
`SEED` fixes the randomness inside XGBoost, so you get the same result each time you run the notebook.

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBRegressor        # the regression version of XGBoost: predicts a number, not a yes/no

warnings.filterwarnings("ignore")              # hide library notices
plt.rcParams["figure.figsize"] = (11, 4)       # default chart size

# Colours used in the charts.
BLUE = "#2a78d6"
ORANGE = "#eb6834"
GREEN = "#1baf7a"
GREY = "#8a8984"

SEED = 42        # same random choices every run
HORIZON = 12     # forecast 12 months ahead

> **Where we are.** The libraries are loaded. First, a look at what's being forecast: monthly benefit payouts, split by
> benefit type.

## 1. The payouts

The file has one row per month per **benefit type** (monthly pension, old age lump sum, unemployment benefit, and so
on), with the amount paid and the number of people paid (`beneficiaries`).

In [ ]:
pay = pd.read_csv("benefit_payouts_monthly.csv", parse_dates=["month"])
pay.head()

That "long" layout (one row per month *and* type) is awkward for charts. `pivot_table` rearranges it into a "wide"
table: **one row per month, one column per benefit type**. Months where a type paid nothing would be empty, so
`fillna(0)` fills them with zero. Dividing by `1e9` gives TZS billions.

In [ ]:
# One row per month, one column per benefit type, in TZS billions.
wide = pay.pivot_table(index="month", columns="benefit_type", values="amount_paid_tzs", aggfunc="sum")
wide = wide.fillna(0) / 1e9      # a type with no payment that month -> 0
wide.tail()

A **stacked area chart** draws each benefit type as a coloured band on top of the previous one, so the top edge of
the chart is the total. It shows both how big the total is and what it's made of.

In [ ]:
# A stacked area chart: each colour is one benefit type, the top edge is the total.
wide.plot.area(stacked=True, figsize=(11, 4.5), lw=0)
plt.title("Benefits paid per month by type (TZS billions)")
plt.ylabel("TZS billions")
plt.legend(loc="upper left", fontsize=8, ncol=2)
plt.show()

A few things to notice before you model anything:

- **Monthly pensions** are the steady base. They grow as more members retire.
- **Old age lump sums** make the total spiky. They're paid in the month a pension claim is approved, and approvals
  come in bunches.
- **Unemployment benefit** only starts in mid-2018, when the benefit was introduced, and rises sharply in 2020.

That last point matters: a model trained on 2017 has never seen an unemployment payment. The structure of the
series changed.

Now make the two series you'll work with: the **total paid** per month (all types added together) and the **number of
beneficiaries** per month. `asfreq("MS")` makes sure there's exactly one row per month (as in Lab 1).

In [ ]:
# The total across all benefit types, and the number of beneficiaries, per month.
total = wide.sum(axis=1).asfreq("MS")                                     # add the columns across each row
beneficiaries = pay.groupby("month")["beneficiaries"].sum().asfreq("MS")  # people paid, all types together

plt.plot(total.index, total.values, color=BLUE)
plt.title("Total benefits paid per month (TZS billions)", loc="left")
plt.show()

> **Where we are.** You've seen that payouts are a steady pension base with spiky lump sums on top, and that a new
> benefit (unemployment) appeared in 2018. You have two monthly series: total paid and beneficiaries. Now you'll turn
> the total into a table XGBoost can learn from.

## 2. Build the feature table

This is the heart of the lab. Each row will be one month; each column something the model can use to predict that
month's payout.

The one rule: every feature must be something you'd **know at the time you make the forecast**. If you're forecasting
next month's payout at the end of this month, you know this month's payout and everything before it. You do
*not* know next month's number of beneficiaries, so beneficiary counts go in only as lags (earlier months' values).

Breaking this rule is called **leakage**, the same problem as Day 2: the model gets a hint it won't have in real use,
looks excellent in testing, and then disappoints.

### What `shift` does

`shift(1)` moves a column down by one row, so each month sits next to the previous month's value:

```
month     payout   lag_1 (= shift(1))   lag_12 (= shift(12))
2024-01    1.82         1.79               1.45  (Jan 2023)
2024-02    1.75         1.82               1.51  (Feb 2023)
2024-03    1.91         1.75               1.60  (Mar 2023)
```

The first rows have nothing to look back on, so their lag columns are empty. That's why the table is passed through
`dropna()` before training.

Try it on the real data and look at three months:

In [ ]:
# See shift on its own.
demo = pd.DataFrame({"payout": total})
demo["lag_1"] = total.shift(1)       # last month's payout
demo["lag_12"] = total.shift(12)     # the payout 12 months ago
demo.loc["2024-01":"2024-03"]

Now build the full table, one group of columns at a time.

**Group 1: lags.** The payout 1, 2, 3, 6 and 12 months earlier. Recent lags tell the model the current level; `lag_12`
tells it what happened in the same month last year (seasonality).

In [ ]:
# The target: this month's payout.
feat = pd.DataFrame({"y": total})

# Lags: the payout 1, 2, 3, 6 and 12 months earlier.
feat["lag_1"] = total.shift(1)
feat["lag_2"] = total.shift(2)
feat["lag_3"] = total.shift(3)
feat["lag_6"] = total.shift(6)
feat["lag_12"] = total.shift(12)

**Group 2: rolling statistics.** A single month can be unusual; an average over several months is steadier.
`rolling(3).mean()` takes the average of each 3-month window. `rolling(6).std()` measures how much the last 6 months
varied: a high value tells the model "things are jumpy right now".

Notice the `shift(1)` first. Without it, the 3-month window for March would include March itself, which is the value
you're trying to predict.

In [ ]:
# Rolling averages of the months BEFORE this one. shift(1) first, so this month is never included.
previous = total.shift(1)
feat["rolling_mean_3"] = previous.rolling(3).mean()      # average of the last 3 months
feat["rolling_mean_6"] = previous.rolling(6).mean()      # ... the last 6
feat["rolling_mean_12"] = previous.rolling(12).mean()    # ... the last 12
feat["rolling_std_6"] = previous.rolling(6).std()        # how much the last 6 months jumped around

**Group 3: calendar and beneficiary features.** The month of the year, the quarter, and a 1/0 flag for June are known
in advance (you always know which month comes next), so they need no shift. Beneficiary counts are not known in
advance, so they go in as lags only.

In [ ]:
# Calendar features: these are known in advance, so no shift needed.
feat["month_of_year"] = feat.index.month                 # 1 to 12
feat["quarter"] = feat.index.quarter                     # 1 to 4
feat["june"] = (feat.index.month == 6).astype(int)       # 1 in June, 0 otherwise

# Beneficiary counts, only as lags (next month's count isn't known yet).
feat["beneficiaries_lag_1"] = beneficiaries.shift(1)
feat["beneficiaries_lag_12"] = beneficiaries.shift(12)

# dropna() removes the first year, whose lag_12 is empty.
feat.dropna().head()

Read one row: for, say, March 2019, `y` is March's payout and every other column describes February and earlier.
The model learns to map "what I knew by the end of February" to "what March turned out to be".

You'll build this same table for other series later in the lab, so wrap the steps above in a function. The `for`
loops do exactly what the cells above did, one lag or window at a time. `beneficiaries=None` makes that argument
optional: some series won't have beneficiary counts.

In [ ]:
def make_features(y, beneficiaries=None):
    """The feature table for a monthly series y: lags, rolling averages of earlier months, calendar columns."""
    table = pd.DataFrame({"y": y})

    # Lags: lag_1, lag_2, lag_3, lag_6, lag_12.
    for lag in [1, 2, 3, 6, 12]:
        table["lag_" + str(lag)] = y.shift(lag)

    # Rolling statistics of the months before this one.
    previous = y.shift(1)
    for window in [3, 6, 12]:
        table["rolling_mean_" + str(window)] = previous.rolling(window).mean()
    table["rolling_std_6"] = previous.rolling(6).std()

    # Calendar columns.
    table["month_of_year"] = table.index.month
    table["quarter"] = table.index.quarter
    table["june"] = (table.index.month == 6).astype(int)

    # Beneficiary lags, only if a beneficiaries series was given.
    if beneficiaries is not None:
        table["beneficiaries_lag_1"] = beneficiaries.shift(1)
        table["beneficiaries_lag_12"] = beneficiaries.shift(12)
    return table


# Check: the function gives exactly the table you built by hand.
print(make_features(total, beneficiaries).equals(feat))

**Your turn (1).** Take out the `.shift(1)` from the rolling features and rebuild the table. Then train a plain
`LinearRegression` (from `sklearn.linear_model`) on both versions, with and without the shift, and compare the
test errors. What happens, and why is the better-looking result misleading?

> **Where we are.** The feature table has one row per month: lags, rolling statistics of earlier months, calendar
> columns and lagged beneficiary counts, and every feature is something you'd know at forecast time. `make_features`
> builds it for any series. Now split it the only honest way for time series.

## 3. Split by time

As in Lab 1, hide 2025 from the model: train on everything up to December 2024, and test on 2025. Never split a time
series at random: the model would then train on months *after* the ones it's tested on, which is impossible in real use.

`X` holds the feature columns, `y` the target. `is_train` is a True/False column: True for months before 2025. `~`
means "not", so `~is_train` picks the 2025 months.

In [ ]:
# Drop the first rows (empty lags), then separate the features (X) from the target (y).
data = feat.dropna()
X = data.drop(columns="y")
y = data["y"]

# Train up to December 2024; test on 2025.
is_train = data.index < "2025-01-01"
X_train = X[is_train]
y_train = y[is_train]
X_test = X[~is_train]        # ~ means "not": the 2025 months
y_test = y[~is_train]

print("train rows:", len(X_train), "from", X_train.index.min().date(), "to", X_train.index.max().date())
print("test rows: ", len(X_test))

About 80 training rows. That's very little for XGBoost, which you'd normally give thousands. With so few rows, a
flexible model can simply **memorise** them (overfitting) and then forecast badly. So you rein it in.

### Theory: regression, and why few rows need a cautious model

On Day 2 XGBoost did **classification**: each leaf of each tree held a score for "irregular". For **regression**, each
leaf holds a number (here, a payout correction in TZS billions), and the trees' outputs are added up to give the
forecast. The learning is the same: each new tree corrects the errors the earlier trees left.

The danger here is the size of the data. With about 80 rows, a flexible model can find a "rule" for almost every
month: *"if lag_3 is between 1.71 and 1.73 and it's April, predict 1.79"*. That fits the past perfectly and means
nothing for the future. The settings in the table below all push the other way: shallow trees, small steps, random
subsets, and a minimum group size. Together they force the model to learn only patterns that appear in many months.

## 4. Train and evaluate: one step ahead

A reminder of how XGBoost works: it builds many small decision trees one after another, each one correcting the
mistakes of the trees before it. The settings below keep it cautious:

| Setting | What it does | Why this value |
|---|---|---|
| `n_estimators=300` | how many trees to build | enough small steps to learn the pattern |
| `max_depth=3` | how many questions each tree may ask | shallow trees can't memorise 80 months |
| `learning_rate=0.03` | how much each tree is allowed to correct | small steps, less overreaction |
| `subsample=0.8` | each tree sees a random 80% of the months | adds variety, reduces memorising |
| `colsample_bytree=0.8` | each tree sees a random 80% of the columns | same idea, for features |
| `min_child_weight=3` | each final group of a tree needs a few months in it | no rules built on a single month |

In [ ]:
# max_depth=3: shallow trees. min_child_weight=3: every final group needs a few months in it.
xgb = XGBRegressor(n_estimators=300, max_depth=3, learning_rate=0.03, subsample=0.8, colsample_bytree=0.8,
                   min_child_weight=3, random_state=SEED)
xgb.fit(X_train, y_train)      # learn from 2017-2024

# Predict each 2025 month from its feature row.
pred_xgb = pd.Series(xgb.predict(X_test), index=X_test.index)
pred_xgb.head()

Score the forecasts with the same three measures as Lab 1 (MAE, RMSE, MAPE; smaller is better):

In [ ]:
def metrics(actual, forecast):
    """MAE, RMSE and MAPE of a forecast (as in Lab 1)."""
    error = actual - forecast                       # one error per month
    mae = error.abs().mean()                        # average size of the error
    rmse = np.sqrt((error ** 2).mean())             # punishes big misses more
    mape = (error.abs() / actual).mean() * 100      # average error as a % of the actual value
    return {"MAE": round(float(mae), 3), "RMSE": round(float(rmse), 3), "MAPE %": round(float(mape), 2)}

Any model must beat simple baselines. Handily, the feature table already contains three of them: `lag_1` *is* the
naive forecast ("same as last month"), `lag_12` *is* the seasonal naive ("same as this month last year"), and
`rolling_mean_3` is "the average of the last three months". So you can score them without any extra work.

In [ ]:
# Three simple forecasts come free from the feature table itself.
one_step = {}
one_step["Naive (last month)"] = metrics(y_test, X_test["lag_1"])
one_step["Seasonal naive (same month last year)"] = metrics(y_test, X_test["lag_12"])
one_step["3-month average"] = metrics(y_test, X_test["rolling_mean_3"])
one_step["XGBoost"] = metrics(y_test, pred_xgb)

# Best (lowest MAE) first.
pd.DataFrame(one_step).T.sort_values("MAE")

This is **one step ahead**: each month's forecast uses the real value from the month before. That's the right
test if finance updates the forecast every month. It's easier than forecasting a whole year at once, which is
next.

**Feature importance** shows which columns the trees used most. It doesn't prove cause and effect, but it's a useful
sanity check: if the model relied mostly on something that shouldn't matter, you'd want to know.

In [ ]:
# Which features did XGBoost use most?
importance = pd.Series(xgb.feature_importances_, index=X_train.columns).sort_values()
importance.tail(10).plot(kind="barh", color=BLUE, figsize=(7, 4))     # the 10 most used, as horizontal bars
plt.title("What XGBoost relied on")
plt.show()

> **Where we are.** One step ahead, XGBoost beats the naive, seasonal naive and 3-month-average forecasts, and the
> feature importance chart shows which columns it leaned on. But one month ahead is the easy case.

### Theory: why longer horizons are harder

One step ahead, every lag the model needs is a real, known value. Twelve steps ahead, most of them aren't known yet:
June's `lag_1` is May, which hasn't happened. You have to fill the gap somehow, and each way has a cost:

```
Recursive:  Jan forecast ─► used as Feb's lag_1 ─► Feb forecast ─► used as Mar's lag_1 ─► ...
            (one model; a January error is carried into every later month)

Direct:     model_1 predicts 1 month ahead
            model_2 predicts 2 months ahead     (twelve models; no errors carried forward,
            ...                                  but each one has less to work with)
            model_12 predicts 12 months ahead
```

Whichever you choose, accuracy falls as the horizon grows, because there's simply less up-to-date information. That's
why forecasting teams report errors **by horizon**, and why finance should update the forecast every month as real
numbers arrive.

## 5. Forecasting twelve months ahead

Finance wants the whole year, not just next month. That's harder: to forecast June 2025 from December 2024, you don't
have `lag_1` (May 2025) yet. There are two standard ways around that with a supervised model:

- **Recursive:** predict next month, pretend it's real, add it to the history, rebuild the features, and predict the
  month after. Repeat twelve times. One model, but errors compound: a mistake in January feeds into February's lags.
- **Direct:** train a separate model for each horizon (one for "1 month ahead", one for "2 months ahead" ...). No
  compounding, but twelve models to maintain.

Here's the recursive version, one month at a time. Read the four numbered steps inside the loop; they repeat 12 times.
`pd.offsets.MonthBegin(1)` moves a date forward to the start of the next month.

In [ ]:
# Start from what we knew at the end of 2024.
history = total[total.index < "2025-01-01"].copy()
history_beneficiaries = beneficiaries[beneficiaries.index < "2025-01-01"].copy()

recursive = {}      # month -> forecast

for step in range(HORIZON):
    # 1. The next month to forecast.
    next_month = history.index[-1] + pd.offsets.MonthBegin(1)

    # 2. Add an empty row for it, so make_features creates its lag columns.
    history.loc[next_month] = np.nan
    # We won't know next month's beneficiaries either: assume the same as 12 months earlier.
    history_beneficiaries.loc[next_month] = history_beneficiaries.iloc[-12]

    # 3. Build the features and keep only the last row (the month we're forecasting).
    row = make_features(history, history_beneficiaries).drop(columns="y").iloc[[-1]]

    # 4. Predict, and write the prediction into the history as if it were real.
    prediction = float(xgb.predict(row)[0])
    history.loc[next_month] = prediction
    recursive[next_month] = prediction

rec = pd.Series(recursive)
rec.round(3)

Compare it with the best baseline from Lab 1, "same month last year, times last year's growth", made from data up to
December 2024 only, so both forecasts know exactly the same things.

In [ ]:
# The baseline from Lab 1: the same month last year, times last year's growth.
before_2025 = total[total.index < "2025-01-01"]
growth = before_2025.iloc[-12:].sum() / before_2025.iloc[-24:-12].sum()     # e.g. 1.10 = 10% growth
snaive_12 = pd.Series(before_2025.iloc[-12:].values * growth, index=rec.index)

print("Seasonal naive + growth:", metrics(y_test, snaive_12))
print("XGBoost (recursive):    ", metrics(y_test, rec))

Draw the actual payouts with three forecasts: recursive (made once, at the end of 2024), the baseline, and one step
ahead (updated every month with the real numbers).

In [ ]:
actual_recent = total.loc["2023":]
plt.plot(actual_recent.index, actual_recent.values, color="black", label="actual")
plt.plot(rec.index, rec.values, color=ORANGE, label="XGBoost, recursive 12 months")
plt.plot(snaive_12.index, snaive_12.values, color=GREY, ls="--", label="seasonal naive + growth")
plt.plot(pred_xgb.index, pred_xgb.values, color=GREEN, alpha=0.8, label="XGBoost, one step ahead")
plt.title("2025 payouts: three ways to forecast", loc="left")
plt.ylabel("TZS billions")
plt.legend()
plt.show()

Notice how the recursive forecast smooths out. The model can't know *when* next year's bunches of lump-sum
approvals will land, so it predicts something close to the average. For a spiky series that's often the best a
forecast can honestly do. It's also a hint that the spiky part (lump sums) might be better forecast separately
from the steady part (monthly pensions). Section 6 does exactly that.

> **Where we are.** The recursive twelve-month forecast is smoother than reality, because the model can't know when
> lump-sum approvals will bunch up, but it still clearly beats the seasonal naive baseline. The main limit is data:
> each series has only about 80 months.

### Theory: global models, and "borrowing strength"

A **local** model learns from one series only. A **global** model learns from many related series at once, with a
column saying which series each row came from.

Why that helps: patterns like "June is high" or "a rising lag_12 means growth" show up in every benefit type. A local
model has to learn them from 80 rows; the global model sees them in 640. Each series **borrows strength** from the
others. The benefit-type column still lets the model learn what's special about each one.

This is now the usual approach for forecasting many related series in retail, energy and finance, and the reason
tree models have become popular for forecasting at all.

## 6. One model for all benefit types

With only about 80 months per series, any single model is data-starved. But you have **eight** series (one per
benefit type), and they share patterns: they all grow, they all react to the same calendar. A **global model** trains
on all of them at once, with the benefit type as an extra feature. Eight series of 80 rows become one table of about
640 rows, so the model has far more examples to learn from.

To build it: make the feature table for each benefit type with `make_features`, add a column naming the type, and stack
the eight tables on top of each other with `pd.concat`. Making `benefit_type` a **category** column tells XGBoost it's
a label (not a number).

In [ ]:
# Build the feature table for each benefit type, add a column saying which type it is, and stack them.
tables = []
for benefit_type in wide.columns:
    table = make_features(wide[benefit_type].asfreq("MS"))
    table["benefit_type"] = benefit_type
    tables.append(table)

panel = pd.concat(tables).dropna()                                  # one tall table
panel["benefit_type"] = panel["benefit_type"].astype("category")    # a label, not a number
print(panel.shape)
panel.head()

Train one model on all types up to 2024. The settings are a little more generous than before (`max_depth=4`, 500 trees)
because there's more data. `tree_method="hist"` is a fast way of building trees that also supports category columns.

In [ ]:
X_panel = panel.drop(columns="y")
y_panel = panel["y"]
is_train = panel.index < "2025-01-01"

# enable_categorical=True lets XGBoost use the benefit_type column directly.
global_model = XGBRegressor(n_estimators=500, max_depth=4, learning_rate=0.03, subsample=0.8, colsample_bytree=0.8,
                            enable_categorical=True, tree_method="hist", random_state=SEED)
global_model.fit(X_panel[is_train], y_panel[is_train])
print("global model trained on", is_train.sum(), "rows")

The global model predicts each benefit type separately. To compare it with the model trained on the total, add the
eight predictions up for each month. `groupby(level=0)` groups by the index, which is the month.

In [ ]:
# Predict every benefit type for 2025, then add the types up per month to get a total.
panel_pred = pd.Series(global_model.predict(X_panel[~is_train]), index=X_panel[~is_train].index)
total_from_parts = panel_pred.groupby(level=0).sum()

print("XGBoost, total series:       ", metrics(y_test, pred_xgb))
print("XGBoost, global model summed:", metrics(y_test, total_from_parts))

Both are one-step-ahead forecasts, so the comparison is fair. Here the global model comes out slightly ahead of the
model trained on the total, even though eight series is a small panel. The idea matters even more when you have
*many* related short series (branches, regions, employer sectors): one model across all of them usually beats one
model per series, because each series borrows strength from the others.

> **Where we are.** You've tested XGBoost one step ahead and twelve steps ahead on the total, and tried a global model
> across all eight benefit types. Time to weigh it all up.

## 7. So, did XGBoost earn its place?

Put the numbers side by side and decide as you would at work. On this series it does:

- One step ahead, it clearly beats last month's value and the 3-month average.
- Twelve months ahead, the recursive forecast roughly halves the error of the seasonal naive. The payout series has
  shifted in structure (unemployment benefit arriving, a growing pension roll), and "same month last year" copes badly
  with that, while the lag features adapt.

Compare that with Lab 1, where the simple baseline won. **Same technique, different data, opposite conclusion.**
That's why you test every time instead of carrying a favourite model from one problem to the next. Lab 3 checks
both conclusions across several years before anyone relies on them.

---

## Looking back, looking ahead

**What you did in this lab.** You rewrote a time series as a supervised-learning table (lags, rolling statistics of
earlier months, calendar columns), split it by time, trained a deliberately cautious XGBoost, beat the baselines one step
ahead, built a recursive twelve-month forecast, and tried one global model across all benefit types.

**The ideas to carry forward.**

- Any time series can become a table of features, but every feature must be known at forecast time. `shift(1)` before
  every rolling window.
- With few rows, rein the model in.
- Forecasts get harder the further ahead you go; say which horizon an error belongs to.
- Related series can share one model and borrow strength from each other.
- The same technique can win on one series and lose on another. Test every time.

**What happens next.** Both of today's conclusions (the baseline won for contributions, XGBoost won for payouts) rest on
a single test year. Lab 3 checks them over several years and adds the ranges finance needs to plan with.

## Answers

**(1)** Without the `shift(1)`, the rolling means include the month being predicted. Build that "leaky" table by
replacing the four rolling columns with unshifted versions:

In [ ]:
from sklearn.linear_model import LinearRegression

# The leaky table: the same features, but the rolling averages now include the current month.
leaky = make_features(total, beneficiaries)
leaky["rolling_mean_3"] = total.rolling(3).mean()
leaky["rolling_mean_6"] = total.rolling(6).mean()
leaky["rolling_mean_12"] = total.rolling(12).mean()
leaky["rolling_std_6"] = total.rolling(6).std()
leaky = leaky.dropna()

# The correct table, for comparison.
honest = feat.dropna()

Train the same simple linear regression on each table, on 2017-2024, and test on 2025:

In [ ]:
# Honest table: train on 2017-2024, test on 2025.
train_h = honest[honest.index < "2025-01-01"]
test_h = honest[honest.index >= "2025-01-01"]
linear_h = LinearRegression().fit(train_h.drop(columns="y"), train_h["y"])
mae_h = metrics(test_h["y"], linear_h.predict(test_h.drop(columns="y")))["MAE"]
print("honest (shifted)     linear regression test MAE:", mae_h)

# Leaky table.
train_l = leaky[leaky.index < "2025-01-01"]
test_l = leaky[leaky.index >= "2025-01-01"]
linear_l = LinearRegression().fit(train_l.drop(columns="y"), train_l["y"])
mae_l = metrics(test_l["y"], linear_l.predict(test_l.drop(columns="y")))["MAE"]
print("leaky (not shifted)  linear regression test MAE:", mae_l)

The leaky version looks almost perfect. The "3-month average" already contains a third of the answer, and a linear
model can simply solve for it: this month = 3 × rolling_mean_3 − lag_1 − lag_2. (Shallow trees on 84 rows happen not to
find that arithmetic, which is exactly why leakage is dangerous: sometimes it shows up, sometimes it hides.) In
production you'd never have that value at forecast time, so the real error would be far worse than reported. Same
lesson as Day 2's `investigation_opened`, in a new costume.